# CLP Graph Time Series: Step-by-Step Anatomy

A guided, exhaustive map of the `graph_Time_series` framework.
It lists every module/function/class and runs tiny examples through the CLP lifecycle: data, augmentation, state, tokens, grammar, residuals, exhaustive search, MCTS, kernels, decomposition, preprocessing, pipeline, and visualization.

Heavy/network-dependent cells are controlled by flags.

## 0. Setup
Adds the repo to `sys.path` and imports the main objects.

In [ ]:
from pathlib import Path
import sys, inspect, importlib, textwrap
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if HERE.name == "graph_Time_series":
    REPO = HERE
elif HERE.parent.name == "graph_Time_series":
    REPO = HERE.parent
elif (HERE / "graph_Time_series").exists():
    REPO = HERE / "graph_Time_series"
else:
    REPO = HERE
for p in [REPO.parent, REPO]:
    p = str(p.resolve())
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO)

In [ ]:
import graph_Time_series as gts
from graph_Time_series import State, Grammar, plot_grammar, mcts_search, print_mcts_tree
from graph_Time_series.token import Token, _shapes
from graph_Time_series.tokens import register_all
from graph_Time_series.tokens.cleaning import CleanIdentity, CleanDetrend, CleanMovingAvg, CleanNormalize, CleanDetrendNorm
from graph_Time_series.tokens.features import FeatRaw, FeatFFTEncode, FeatLagFeatures
from graph_Time_series.tokens.models import ModelKernelRBF, ModelRandomForest, ModelXGBoost, StopToken, compute_mase
from graph_Time_series.heuristics import compute_mi_score, compute_action_priors

np.set_printoptions(precision=4, suppress=True)

## 1. Complete Public Inventory
This is the coverage checklist: every function and class found in the package modules.

In [ ]:
MODULES = [
    "augmentation", "config", "data", "decomposition", "grammar", "heuristics",
    "kernels", "mcts", "pipeline", "preprocessing", "state", "token", "viz",
    "tokens.cleaning", "tokens.features", "tokens.models", "tokens",
]

def load_module(name):
    return importlib.import_module(f"graph_Time_series.{name}")

def show_module_inventory(module_name):
    mod = load_module(module_name)
    print(f"\n# graph_Time_series.{module_name}")
    for name, obj in vars(mod).items():
        keep_private = name in {"_shapes", "_dtw_cost", "_dtw_distance_matrix", "_morlet_wavelet", "_ricker_wavelet", "_shift_mask", "_load_hf"}
        if name.startswith("_") and not keep_private:
            continue
        if inspect.isclass(obj) and getattr(obj, "__module__", "").startswith("graph_Time_series"):
            print("class", name)
        elif inspect.isfunction(obj) and getattr(obj, "__module__", "").startswith("graph_Time_series"):
            try: sig = inspect.signature(obj)
            except Exception: sig = "(?)"
            print("def  ", f"{name}{sig}")

for m in MODULES:
    show_module_inventory(m)

## 2. Synthetic Dataset
The CLP state expects arrays H=(n_samples, history_len), F=(n_samples, horizon).

In [ ]:
def make_toy_dataset(n=12, history_len=48, horizon=12, seed=0):
    rng = np.random.default_rng(seed)
    th = np.arange(history_len)
    tf = np.arange(history_len, history_len + horizon)
    H, F = [], []
    for _ in range(n):
        amp = rng.uniform(0.8, 1.6); phase = rng.uniform(0, np.pi)
        trend = rng.uniform(-0.02, 0.04); offset = rng.uniform(5, 12)
        yh = offset + trend * th + amp * np.sin(2*np.pi*th/24 + phase)
        yf = offset + trend * tf + amp * np.sin(2*np.pi*tf/24 + phase)
        H.append(yh + rng.normal(0, .05, size=history_len))
        F.append(yf + rng.normal(0, .05, size=horizon))
    return np.asarray(H, dtype=np.float32), np.asarray(F, dtype=np.float32)

H, F = make_toy_dataset()
print(H.shape, F.shape)
plt.figure(figsize=(8,3))
plt.plot(np.arange(-H.shape[1],0), H[0], label="history")
plt.plot(np.arange(F.shape[1]), F[0], label="future")
plt.axvline(0, ls=":", color="tab:blue"); plt.legend(); plt.grid(alpha=.25);

## 3. Data Module
Purpose: GIFT-Eval loading, cleaning, config resolution, normalizing and preparing examples.

In [ ]:
from graph_Time_series import data as data_mod
DATA_FUNCS = ["clean_series", "extract_history_future", "normalize_by_history", "resample_series", "resolve_config", "list_pretrain_subsets", "_load_hf", "load_gift_dataset", "quick_peek", "dataset_summary", "build_examples", "prepare_examples"]
for name in DATA_FUNCS:
    obj = getattr(data_mod, name)
    print(f"\n{name}{inspect.signature(obj)}")
    print((inspect.getdoc(obj) or "").split("\n")[0])

x_dirty = np.array([1., np.nan, 3., np.inf, 5.])
print("clean_series:", data_mod.clean_series(x_dirty))
hn, fn, mu, sigma = data_mod.normalize_by_history(np.array([1,2,3.]), np.array([4,5.]))
print("normalize_by_history:", hn, fn, mu, sigma)
print("resample:", data_mod.resample_series(np.array([1,3,5.]), 7))

## 4. Augmentation Module
Purpose: shift, smooth, jitter, crop/pad, uniform length, and prepared-record augmentation. Leakage rule: do not move future values into history.

In [ ]:
from graph_Time_series import augmentation as aug
AUG_FUNCS = ["apply_shift", "smooth", "jitter", "crop_or_pad", "uniform_length", "_shift_mask", "augment_train"]
for name in AUG_FUNCS:
    obj = getattr(aug, name)
    print(f"\n{name}{inspect.signature(obj)}")
    print((inspect.getdoc(obj) or "").split("\n")[0])

x = H[0]
plt.figure(figsize=(9,3))
plt.plot(x, label="original")
plt.plot(aug.apply_shift(x, 4), label="shift +4")
plt.plot(aug.smooth(x, 7), label="smooth 7")
plt.plot(aug.jitter(x, sigma=.03, rng=np.random.default_rng(1)), label="jitter")
plt.legend(); plt.grid(alpha=.25);

## 5. State Object
Purpose: mutable rollout state. It stores raw data, features, predictions, residual target, logs, metadata, and termination status.

In [ ]:
s = State(H, F)
print(s)
print("features", s.features.keys())
print("target", s.current_target.shape, np.linalg.norm(s.current_target))
print("last_token", s.last_token, "depth", s.depth, "models", s.n_models_applied)
print(inspect.getsource(State.push_prediction))

## 6. Token Base Class
Purpose: all computational steps inherit from Token and implement apply(state).

In [ ]:
print(inspect.getsource(Token))
print(inspect.getsource(_shapes))

## 7. Cleaning Tokens
Purpose: read raw_history and create cleaned histories. Normalize also scales current_target.

In [ ]:
cleaning_tokens = [CleanIdentity(), CleanDetrend(), CleanMovingAvg(), CleanNormalize(), CleanDetrendNorm()]
for tok in cleaning_tokens:
    st = State(H, F)
    print(f"\n{tok.name}: reads={tok.reads} writes={tok.writes}")
    print(tok.description)
    tok.apply(st)
    print("cleaned", st.features["cleaned"].shape, "normalized", st.metadata.get("normalized", False), "target_norm", np.linalg.norm(st.current_target))

## 8. Feature Tokens
Purpose: read cleaned histories and write model_input.

In [ ]:
feature_tokens = [FeatRaw(), FeatFFTEncode(), FeatLagFeatures()]
base = State(H, F); CleanNormalize().apply(base)
for tok in feature_tokens:
    st = base.copy(); tok.apply(st)
    print(f"{tok.name:12s}", st.features["model_input"].shape)

## 9. Model and STOP Tokens
Purpose: models fit LOO predictions to current_target; STOP sums predictions, unnormalizes, clips and computes MASE.

In [ ]:
from graph_Time_series.tokens import models as model_mod
for name in ["squared_distance_matrix", "median_heuristic_lengthscale", "compute_mase"]:
    obj = getattr(model_mod, name)
    print(f"\n{name}{inspect.signature(obj)}")
    print((inspect.getdoc(obj) or "").split("\n")[0])

for tok in [ModelKernelRBF(), ModelRandomForest(), ModelXGBoost(), StopToken()]:
    print(f"\n{tok.name} [{tok.token_class}] reads={tok.reads} writes={tok.writes}")
    print(tok.description)

### Scale-Aware Residual Patch
If targets are normalized, subsequent residuals must stay normalized. This patch is recommended before sequential normalized model chains.

In [ ]:
def scale_aware_push_prediction(self, pred: np.ndarray, token_name: str):
    self.prediction_stack.append(pred)
    self.prediction_names.append(token_name)
    target_base = self.original_future
    if self.metadata.get("normalized"):
        mu = self.features.get("norm_mu", 0.0)
        sigma = self.features.get("norm_sigma", 1.0)
        target_base = (self.original_future - mu) / sigma
    self.current_target = target_base - self.cumulative_prediction()

State.push_prediction = scale_aware_push_prediction
print("patched State.push_prediction for scale-aware residual chaining")

## 10. Residual Chaining Demo
Model 2 receives the residual left by model 1. With the scale-aware patch, the residual should not explode after normalization.

In [ ]:
st = State(H, F)
CleanNormalize().apply(st)
FeatLagFeatures().apply(st)
print("after normalize", np.linalg.norm(st.current_target))
ModelKernelRBF().apply(st)
print("after model 1", np.linalg.norm(st.current_target))
ModelKernelRBF().apply(st)
print("after model 2", np.linalg.norm(st.current_target))
StopToken().apply(st)
print("prediction stack", st.prediction_names)
print("MASE", st.mase)

## 11. Grammar
Purpose: directed graph of valid token transitions plus feature availability checks.

In [ ]:
g = Grammar(); register_all(g)
print(g)
print("valid at START", g.valid_actions(State(H, F)))
st = State(H, F); CleanNormalize().apply(st)
print("valid after normalize", g.valid_actions(st))
plot_grammar(g, title="Default grammar")
print(inspect.getsource(Grammar.valid_actions))

## 12. Heuristics
Purpose: mutual-information score and action priors for PUCT/MCTS.

In [ ]:
st = State(H, F); CleanNormalize().apply(st); FeatLagFeatures().apply(st)
mi = compute_mi_score(st)
actions = ["kernel_rbf", "random_forest", "xgboost", "STOP"]
print("MI", mi)
pprint(compute_action_priors(mi, actions))

## 13. MCTS
Purpose: grammar-guided search using PUCT and reward = 1 / (1 + MASE).

In [ ]:
from graph_Time_series.mcts import MCTSNode
print(inspect.getsource(MCTSNode))
print("mcts_search", inspect.signature(mcts_search))

RUN_TINY_MCTS = True
if RUN_TINY_MCTS:
    res = mcts_search(g, State(H[:8], F[:8]), n_iterations=5, verbose=True)
    print(res["best_chain"], res["best_mase"])
    print_mcts_tree(res["root"], max_depth=4)

## 14. Exhaustive Search Pattern
Useful while the grammar is small and you want to inspect all combinations.

In [ ]:
from itertools import product

def run_chain(chain, H=H, F=F):
    st = State(H, F)
    for name in chain:
        g.tokens[name].apply(st)
    if not st.terminated:
        g.tokens["STOP"].apply(st)
    return st

chains=[]
for cleaner in ["normalize", "detrend_norm"]:
    for feature in ["feat_raw", "fft_encode", "feat_lag"]:
        for depth in [1,2]:
            for models in product(["kernel_rbf"], repeat=depth):
                chains.append([cleaner, feature, *models, "STOP"])

rows=[]
for chain in chains:
    try:
        st = run_chain(chain, H[:8], F[:8])
        rows.append((st.mase, " -> ".join(chain)))
    except Exception as exc:
        rows.append((np.inf, " -> ".join(chain) + " ERROR " + repr(exc)))

for mv, chain in sorted(rows)[:10]:
    print(f"MASE={mv:8.4f} | {chain}")

## 15. Kernels Module
Purpose: standalone kernel-operator regression layer.

In [ ]:
mod = importlib.import_module("graph_Time_series.kernels")
NAMES = ["Kernel","squared_distance_matrix","median_heuristic_lengthscale","RBFKernel","build_sample_mask","build_mask_matrix","MaskedRBFKernel","LinearKernel","WaveletKernel","_dtw_cost","_dtw_distance_matrix","NTKKernel","RFMKernel","ConvRFMKernel","DTWKernel","rbf_kernel","fit_kernel_operator","predict_kernel_operator","rmse","relative_rmse","mase"]
for name in NAMES:
    obj = getattr(mod, name)
    try: sig = inspect.signature(obj)
    except Exception: sig = ""
    print(f"{name}{sig}")
    doc = (inspect.getdoc(obj) or "").split("\n")[0]
    if doc: print("   ", doc)

## 16. Decomposition Module
Purpose: EMD/KMD/CWT signal decomposition and feature builders.

In [ ]:
mod = importlib.import_module("graph_Time_series.decomposition")
NAMES = ["emd_decompose","kmd_decompose","cwt_decompose","decompose_series","make_decomposition_feature_fn"]
for name in NAMES:
    obj = getattr(mod, name)
    try: sig = inspect.signature(obj)
    except Exception: sig = ""
    print(f"{name}{sig}")
    doc = (inspect.getdoc(obj) or "").split("\n")[0]
    if doc: print("   ", doc)

## 17. Preprocessing Module
Purpose: slice long pretrain series into history/future chunks.

In [ ]:
mod = importlib.import_module("graph_Time_series.preprocessing")
NAMES = ["slice_series","slice_pretrain","slice_domain"]
for name in NAMES:
    obj = getattr(mod, name)
    try: sig = inspect.signature(obj)
    except Exception: sig = ""
    print(f"{name}{sig}")
    doc = (inspect.getdoc(obj) or "").split("\n")[0]
    if doc: print("   ", doc)

## 18. Pipeline Module
Purpose: original kernel train/eval workflow independent of CLP tokens.

In [ ]:
mod = importlib.import_module("graph_Time_series.pipeline")
NAMES = ["TrainEvalResult","train_eval","run_all_domains","ScalingResult","scaling_study"]
for name in NAMES:
    obj = getattr(mod, name)
    try: sig = inspect.signature(obj)
    except Exception: sig = ""
    print(f"{name}{sig}")
    doc = (inspect.getdoc(obj) or "").split("\n")[0]
    if doc: print("   ", doc)

## 19. Visualization Module
Purpose: plotting helpers.

In [ ]:
mod = importlib.import_module("graph_Time_series.viz")
NAMES = ["plot_series","plot_series_grid","plot_distribution","plot_train_test_split","plot_kernel_heatmap","plot_residuals","plot_metric_summary","plot_predictions_for_domain","report_eval","compare_results"]
for name in NAMES:
    obj = getattr(mod, name)
    try: sig = inspect.signature(obj)
    except Exception: sig = ""
    print(f"{name}{sig}")
    doc = (inspect.getdoc(obj) or "").split("\n")[0]
    if doc: print("   ", doc)

## 20. Safe Demos for Large Modules
Small demos for kernels, CWT, slicing, and visualization.

In [ ]:
from graph_Time_series import kernels as km, decomposition as decomp, preprocessing as prep, viz

x_train = H[:6, -24:]; y_train = F[:6]; x_query = H[6:8, -24:]
model = km.fit_kernel_operator(x_train, y_train, gamma=1e-2, rng=np.random.default_rng(0), kernel=km.RBFKernel())
pred = km.predict_kernel_operator(model, x_query)
print("kernel pred", pred.shape, "rmse", km.rmse(F[6:8], pred))

cwt = decomp.cwt_decompose(H[0], n_scales=8, quiet=False)
print("cwt", {k: np.shape(v) for k,v in cwt.items()})

series = np.sin(np.arange(300)/12) + np.arange(300)*.001
chunks = prep.slice_series(series, future_len=24, min_history=60, max_history=80, min_future=12, seed=0)
print("chunks", len(chunks), chunks[0]["history"].shape, chunks[0]["future"].shape)

viz.plot_series(H[0], F[0], title="plot_series demo", show=True)

## 21. Leakage Checklist

- Feature tokens should not read future, original_future, current_target, or residuals.
- Forecasting augmentation should not move future values into history.
- If targets are normalized, residual updates must remain normalized until STOP.
- STOP is the only place that should compare forecast to ground truth.
- LOO models must not train on the held-out sample.
- Augmented copies can make LOO optimistic if near-duplicates remain in the training fold. For final evaluation, split originals first, augment training only, evaluate on untouched originals.

## 22. Where to Add New Ideas

- New cleaner: tokens/cleaning.py, then tokens/__init__.py.
- New feature extractor: tokens/features.py.
- New model/residual learner: tokens/models.py.
- New search prior: heuristics.py.
- New grammar policy: register_all or a notebook-local grammar builder.
- New low-level kernel: kernels.py.
- New decomposition feature: decomposition.py or a token that wraps it.